# GR-CS-HJEPA Chapter 4 — Phase 1B Remediation and Gated Confirmatory Launch

This notebook is the next executable step after the successful Phase 1 review-freeze run. It is designed to answer the immediate research gate:

> Can the H-JEPA representation become **noncollapsed** and **useful enough** that small frozen-backbone downstream heads can read task structure, and can routing metrics be coupled to learned model weights?

Only if those gates pass will the notebook write a confirmatory launch plan for the frozen protocol.

**Important:** Phase 1B is still a remediation/pilot phase. It is not a dissertation confirmatory result. The notebook defaults to a safe mode that does not execute expensive confirmatory seeds automatically.

## 0. Runtime policy

Use this notebook in Colab Pro+ or a local Jupyter environment. Colab runtimes are temporary, so keep GitHub as source of truth and Google Drive as the artifact store. The notebook will write a Python package under `/content/grcshjepa_phase1b` unless you change `PROJECT_DIR`.

In [ ]:
# === Runtime controls ===
PROJECT_DIR = "/content/grcshjepa_phase1b"
MOUNT_GOOGLE_DRIVE = True
ARCHIVE_TO_DRIVE = True

# Use configs/phase1b_quick.yaml for a short dry run; use phase1b_remediation.yaml for the intended Phase 1B run.
PHASE1B_CONFIG = "configs/phase1b_remediation.yaml"
# PHASE1B_CONFIG = "configs/phase1b_quick.yaml"

# Confirmatory launch safety controls.
# The notebook will only write/run launch commands if Phase 1B gates pass.
AUTO_LAUNCH_CONFIRMATORY = False
REAL_CONFIRMATORY_RUN = False  # Keep False unless production confirmatory runners are locked.

import os, sys, json, textwrap, subprocess, pathlib, shutil, time
from pathlib import Path

print("Project dir:", PROJECT_DIR)

In [ ]:
# === Optional Google Drive mount ===
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted.")
    except Exception as exc:
        print("Drive mount skipped or unavailable:", repr(exc))

## 1. Write the tested project scaffold

The next cell writes an installable Python package with:

- strengthened anti-collapse loss with variance hinge;
- nontrivial maze and sorting context-target datasets;
- H-JEPA model with stop-gradient EMA target encoder;
- frozen-backbone downstream-head training;
- learned-weight-coupled routing metrics;
- Phase 1B gate review; and
- gated confirmatory protocol launch-plan generation.

In [ ]:
# === Write project files ===
from pathlib import Path
import json, textwrap, os, shutil
PROJECT = Path(PROJECT_DIR)
PROJECT.mkdir(parents=True, exist_ok=True)
files = json.loads('{"pyproject.toml": "[build-system]\\nrequires = [\\"setuptools\\", \\"wheel\\"]\\nbuild-backend = \\"setuptools.build_meta\\"\\n\\n[project]\\nname = \\"grcshjepa\\"\\nversion = \\"0.2.0\\"\\ndescription = \\"GR-CS-HJEPA Phase 1B remediation and gated confirmatory launch pipeline\\"\\nrequires-python = \\">=3.10\\"\\ndependencies = [\\n    \\"numpy\\",\\n    \\"pandas\\",\\n    \\"pyyaml\\",\\n    \\"torch\\",\\n    \\"tqdm\\",\\n]\\n\\n[project.optional-dependencies]\\ndev = [\\"pytest\\"]\\n\\n[tool.setuptools.packages.find]\\nwhere = [\\"src\\"]\\n\\n[tool.ruff]\\nline-length = 100\\n\\n[tool.black]\\nline-length = 100\\n", "README.md": "# GR-CS-HJEPA Chapter 4 Phase 1B Remediation\\n\\nThis project is a Colab-ready, tested scaffold for the **Phase 1B remediation** step after Phase 1 review-freeze placed the confirmatory protocol on `conditional_hold_not_launchable`.\\n\\nThe notebook and package implement the next research gate:\\n\\n1. strengthen anti-collapse regularization with first/second-moment matching plus a variance hinge;\\n2. use nontrivial context-target construction for maze and sorting predictive representation learning;\\n3. train small downstream heads on a frozen H-JEPA backbone;\\n4. verify that those heads beat minimum readiness gates and naive baselines;\\n5. couple routing metrics to learned model weights rather than a disconnected toy graph;\\n6. review all launch gates; and\\n7. launch the frozen confirmatory protocol only if all gates pass.\\n\\nThe default notebook does **not** auto-run expensive confirmatory seeds. It creates a dry-run launch plan. Set `AUTO_LAUNCH_CONFIRMATORY = True` and `REAL_CONFIRMATORY_RUN = True` only after the Phase 1B gate memo says `launchable_confirmatory_ready`.\\n\\nPhase 1B results are still pilot/remediation evidence, not dissertation confirmatory findings.\\n", "src/grcshjepa/__init__.py": "\\"\\"\\"GR-CS-HJEPA Phase 1B remediation package.\\"\\"\\"\\n\\n__all__ = [\\"phase1b\\"]\\n__version__ = \\"0.2.0\\"\\n", "src/grcshjepa/phase1b.py": "from __future__ import annotations\\n\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nimport platform\\nimport random\\nimport subprocess\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nfrom typing import Any, Dict, Iterable, List, Optional, Tuple\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nimport yaml\\nfrom torch.utils.data import DataLoader, TensorDataset\\n\\n\\n# -----------------------------\\n# Reproducibility utilities\\n# -----------------------------\\n\\n\\ndef set_seed(seed: int) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.backends.cudnn.deterministic = False\\n    torch.backends.cudnn.benchmark = True\\n\\n\\ndef git_commit_hash() -> str:\\n    try:\\n        return subprocess.check_output([\\"git\\", \\"rev-parse\\", \\"HEAD\\"], text=True).strip()\\n    except Exception:\\n        return \\"unknown\\"\\n\\n\\ndef save_json(path: str | Path, payload: Dict[str, Any]) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with path.open(\\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(payload, f, indent=2, sort_keys=True)\\n\\n\\ndef load_yaml(path: str | Path) -> Dict[str, Any]:\\n    with Path(path).open(\\"r\\", encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f)\\n\\n\\ndef save_yaml(path: str | Path, payload: Dict[str, Any]) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with path.open(\\"w\\", encoding=\\"utf-8\\") as f:\\n        yaml.safe_dump(payload, f, sort_keys=False)\\n\\n\\ndef config_hash(payload: Dict[str, Any]) -> str:\\n    s = json.dumps(payload, sort_keys=True, default=str)\\n    return hashlib.sha256(s.encode(\\"utf-8\\")).hexdigest()[:16]\\n\\n\\ndef write_manifest(path: str | Path, payload: Dict[str, Any]) -> None:\\n    payload = {\\n        \\"written_at_unix\\": time.time(),\\n        \\"git_commit\\": git_commit_hash(),\\n        \\"python\\": platform.python_version(),\\n        \\"platform\\": platform.platform(),\\n        \\"torch_version\\": torch.__version__,\\n        \\"cuda_available\\": torch.cuda.is_available(),\\n        \\"device_name\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\"cpu\\",\\n        **payload,\\n    }\\n    save_json(path, payload)\\n\\n\\n@dataclass\\nclass Phase1BConfig:\\n    output_dir: str = \\"runs/phase1b_remediation\\"\\n    seeds: Tuple[int, ...] = (0, 1, 2, 3, 4, 5)\\n    device: str = \\"auto\\"\\n    maze_size: int = 8\\n    sort_length: int = 10\\n    train_n: int = 768\\n    val_n: int = 256\\n    test_n: int = 256\\n    batch_size: int = 64\\n    latent_dim: int = 64\\n    projection_dim: int = 32\\n    hidden_dim: int = 128\\n    epochs: int = 14\\n    head_epochs: int = 25\\n    learning_rate: float = 2.0e-3\\n    head_learning_rate: float = 2.0e-3\\n    weight_decay: float = 1.0e-4\\n    ema_tau: float = 0.995\\n    lambda_ac: float = 0.50\\n    variance_floor: float = 0.70\\n    lambda_var: float = 2.0\\n    low_shot_fraction: float = 0.30\\n    max_horizon: int = 4\\n    # launch gates, intentionally copied into config to avoid hidden thresholds\\n    gate_effective_rank_min: float = 6.0\\n    gate_cov_trace_min: float = 1.0\\n    gate_ac_loss_max: float = 8.0\\n    gate_maze_action_acc_min: float = 0.35\\n    gate_sort_mse_max: float = 0.12\\n    gate_sort_exactish_min: float = 0.05\\n    gate_routing_damage_drop_min: float = 1.0e-8\\n\\n    @classmethod\\n    def from_yaml(cls, path: str | Path) -> \\"Phase1BConfig\\":\\n        data = load_yaml(path)\\n        if \\"seeds\\" in data and isinstance(data[\\"seeds\\"], list):\\n            data[\\"seeds\\"] = tuple(data[\\"seeds\\"])\\n        return cls(**data)\\n\\n    def to_dict(self) -> Dict[str, Any]:\\n        d = asdict(self)\\n        d[\\"seeds\\"] = list(self.seeds)\\n        return d\\n\\n\\ndef resolve_device(device: str) -> torch.device:\\n    if device == \\"auto\\":\\n        return torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n    if device == \\"cuda\\" and not torch.cuda.is_available():\\n        print(\\"Requested CUDA but CUDA is unavailable; falling back to CPU.\\")\\n        return torch.device(\\"cpu\\")\\n    return torch.device(device)\\n\\n\\n# -----------------------------\\n# Diagnostics and losses\\n# -----------------------------\\n\\n\\ndef covariance_matrix(y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:\\n    if y.ndim != 2:\\n        raise ValueError(f\\"Expected [batch, dim] tensor, got {tuple(y.shape)}\\")\\n    centered = y - y.mean(dim=0, keepdim=True)\\n    denom = max(y.shape[0] - 1, 1)\\n    return centered.T @ centered / denom + eps * torch.eye(y.shape[1], device=y.device, dtype=y.dtype)\\n\\n\\ndef effective_rank_from_cov(cov: torch.Tensor, eps: float = 1e-12) -> float:\\n    eig = torch.linalg.eigvalsh(cov).clamp_min(0)\\n    total = eig.sum()\\n    if float(total.detach().cpu()) <= eps:\\n        return 0.0\\n    p = eig / (total + eps)\\n    entropy = -(p * (p + eps).log()).sum()\\n    return float(torch.exp(entropy).detach().cpu())\\n\\n\\ndef anti_collapse_loss(\\n    y: torch.Tensor,\\n    variance_floor: float = 0.70,\\n    lambda_var: float = 2.0,\\n    eps: float = 1e-6,\\n) -> Tuple[torch.Tensor, Dict[str, float]]:\\n    \\"\\"\\"Moment-based anti-collapse loss with explicit variance hinge.\\n\\n    The loss is intentionally normalized by dimension so that the launch threshold is interpretable.\\n    It penalizes a nonzero batch mean, dead coordinates, and redundant covariance structure.\\n    \\"\\"\\"\\n    if y.ndim != 2:\\n        raise ValueError(f\\"Expected [batch, dim] tensor, got {tuple(y.shape)}\\")\\n    b, d = y.shape\\n    mean = y.mean(dim=0)\\n    centered = y - mean\\n    cov = centered.T @ centered / max(b - 1, 1)\\n    diag = torch.diag(cov)\\n    std = torch.sqrt(diag + eps)\\n    eye = torch.eye(d, device=y.device, dtype=y.dtype)\\n    offdiag = cov - torch.diag(diag)\\n\\n    mean_loss = mean.pow(2).mean()\\n    diag_loss = (diag - 1.0).pow(2).mean()\\n    offdiag_loss = offdiag.pow(2).sum() / max(d, 1)\\n    var_hinge = F.relu(variance_floor - std).pow(2).mean()\\n    loss = mean_loss + diag_loss + offdiag_loss + lambda_var * var_hinge\\n\\n    with torch.no_grad():\\n        eig = torch.linalg.eigvalsh(cov + eps * eye).clamp_min(0)\\n        stats = {\\n            \\"ac_loss\\": float(loss.detach().cpu()),\\n            \\"ac_mean_loss\\": float(mean_loss.detach().cpu()),\\n            \\"ac_diag_loss\\": float(diag_loss.detach().cpu()),\\n            \\"ac_offdiag_loss\\": float(offdiag_loss.detach().cpu()),\\n            \\"ac_var_hinge\\": float(var_hinge.detach().cpu()),\\n            \\"effective_rank\\": effective_rank_from_cov(cov + eps * eye),\\n            \\"cov_trace\\": float(torch.trace(cov).detach().cpu()),\\n            \\"cov_min_eig\\": float(eig.min().detach().cpu()),\\n            \\"cov_max_eig\\": float(eig.max().detach().cpu()),\\n            \\"min_std\\": float(std.min().detach().cpu()),\\n            \\"mean_std\\": float(std.mean().detach().cpu()),\\n        }\\n    return loss, stats\\n\\n\\ndef normalized_prediction_loss(z_hat: torch.Tensor, z_tgt: torch.Tensor) -> torch.Tensor:\\n    z_hat_n = F.normalize(z_hat, dim=-1)\\n    z_tgt_n = F.normalize(z_tgt, dim=-1)\\n    return (z_hat_n - z_tgt_n).pow(2).mean()\\n\\n\\n# -----------------------------\\n# Synthetic tasks\\n# -----------------------------\\n\\n\\ndef bfs_path(grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int]) -> List[Tuple[int, int]]:\\n    from collections import deque\\n\\n    h, w = grid.shape\\n    q = deque([start])\\n    parent: Dict[Tuple[int, int], Optional[Tuple[int, int]]] = {start: None}\\n    while q:\\n        r, c = q.popleft()\\n        if (r, c) == goal:\\n            break\\n        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\\n            nr, nc = r + dr, c + dc\\n            nxt = (nr, nc)\\n            if 0 <= nr < h and 0 <= nc < w and grid[nr, nc] == 0 and nxt not in parent:\\n                parent[nxt] = (r, c)\\n                q.append(nxt)\\n    if goal not in parent:\\n        return []\\n    path: List[Tuple[int, int]] = []\\n    cur: Optional[Tuple[int, int]] = goal\\n    while cur is not None:\\n        path.append(cur)\\n        cur = parent[cur]\\n    return list(reversed(path))\\n\\n\\ndef distance_to_goal(grid: np.ndarray, goal: Tuple[int, int]) -> np.ndarray:\\n    from collections import deque\\n\\n    h, w = grid.shape\\n    dist = np.full((h, w), h * w, dtype=np.float32)\\n    q = deque([goal])\\n    dist[goal] = 0\\n    while q:\\n        r, c = q.popleft()\\n        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\\n            nr, nc = r + dr, c + dc\\n            if 0 <= nr < h and 0 <= nc < w and grid[nr, nc] == 0:\\n                if dist[nr, nc] > dist[r, c] + 1:\\n                    dist[nr, nc] = dist[r, c] + 1\\n                    q.append((nr, nc))\\n    dist[grid == 1] = h * w\\n    return dist / max(1.0, float(h * w))\\n\\n\\ndef make_maze_dataset(n: int, seed: int, size: int = 8, obstacle_prob: float = 0.18) -> TensorDataset:\\n    rng = np.random.default_rng(seed)\\n    xs, targets, actions = [], [], []\\n    action_counts = {0: 0, 1: 0, 2: 0, 3: 0}\\n    max_per_action = int(math.ceil(n / 4))\\n    while len(xs) < n:\\n        grid = (rng.random((size, size)) < obstacle_prob).astype(np.float32)\\n        # Randomize start/goal to avoid a majority-action shortcut. Require nontrivial distance.\\n        start = (int(rng.integers(0, size)), int(rng.integers(0, size)))\\n        goal = (int(rng.integers(0, size)), int(rng.integers(0, size)))\\n        if start == goal or abs(start[0] - goal[0]) + abs(start[1] - goal[1]) < max(3, size // 2):\\n            continue\\n        grid[start] = 0.0\\n        grid[goal] = 0.0\\n        path = bfs_path(grid, start, goal)\\n        if len(path) < 2:\\n            continue\\n        path_mask = np.zeros_like(grid, dtype=np.float32)\\n        for r, c in path:\\n            path_mask[r, c] = 1.0\\n        start_ch = np.zeros_like(grid, dtype=np.float32); start_ch[start] = 1.0\\n        goal_ch = np.zeros_like(grid, dtype=np.float32); goal_ch[goal] = 1.0\\n        dist_ch = distance_to_goal(grid, goal)\\n        # Context has no path channel. Target reveals the solver-derived path channel.\\n        x = np.stack([grid, start_ch, goal_ch, dist_ch, np.zeros_like(grid)], axis=0)\\n        tgt = np.stack([grid, start_ch, goal_ch, dist_ch, path_mask], axis=0)\\n        r1, c1 = path[0]\\n        r2, c2 = path[1]\\n        if r2 > r1:\\n            act = 1  # down\\n        elif r2 < r1:\\n            act = 0  # up\\n        elif c2 > c1:\\n            act = 3  # right\\n        else:\\n            act = 2  # left\\n        if action_counts[act] >= max_per_action:\\n            continue\\n        xs.append(x.reshape(-1))\\n        targets.append(tgt.reshape(-1))\\n        actions.append(act)\\n        action_counts[act] += 1\\n    return TensorDataset(\\n        torch.tensor(np.stack(xs), dtype=torch.float32),\\n        torch.tensor(np.stack(targets), dtype=torch.float32),\\n        torch.tensor(np.array(actions), dtype=torch.long),\\n    )\\n\\n\\ndef bubble_step(seq: np.ndarray) -> np.ndarray:\\n    arr = seq.copy()\\n    for i in range(len(arr) - 1):\\n        if arr[i] > arr[i + 1]:\\n            arr[i], arr[i + 1] = arr[i + 1], arr[i]\\n            return arr\\n    return arr\\n\\n\\ndef inversion_count(seq: np.ndarray) -> int:\\n    return int(sum(seq[i] > seq[j] for i in range(len(seq)) for j in range(i + 1, len(seq))))\\n\\n\\ndef make_sorting_dataset(n: int, seed: int, length: int = 10, max_horizon: int = 4) -> TensorDataset:\\n    rng = np.random.default_rng(seed)\\n    xs, targets, sorted_targets, horizons = [], [], [], []\\n    for _ in range(n):\\n        seq = rng.permutation(length).astype(np.float32) / max(1, length - 1)\\n        horizon = int(rng.integers(1, max_horizon + 1))\\n        future = seq.copy()\\n        for _ in range(horizon):\\n            future = bubble_step(future)\\n        sorted_seq = np.sort(seq)\\n        inv = inversion_count(seq)\\n        # context appends simple self-supervised structure features, not labels from a classifier.\\n        feats = np.array([inv / (length * (length - 1) / 2), horizon / max_horizon], dtype=np.float32)\\n        x = np.concatenate([seq, feats, np.zeros(length, dtype=np.float32)], axis=0)\\n        tgt = np.concatenate([future, feats, sorted_seq], axis=0)\\n        xs.append(x)\\n        targets.append(tgt)\\n        sorted_targets.append(sorted_seq)\\n        horizons.append(horizon)\\n    return TensorDataset(\\n        torch.tensor(np.stack(xs), dtype=torch.float32),\\n        torch.tensor(np.stack(targets), dtype=torch.float32),\\n        torch.tensor(np.stack(sorted_targets), dtype=torch.float32),\\n        torch.tensor(np.array(horizons), dtype=torch.long),\\n    )\\n\\n\\n# -----------------------------\\n# Model\\n# -----------------------------\\n\\n\\nclass Encoder(nn.Module):\\n    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int):\\n        super().__init__()\\n        self.net = nn.Sequential(\\n            nn.Linear(input_dim, hidden_dim),\\n            nn.GELU(),\\n            nn.LayerNorm(hidden_dim),\\n            nn.Linear(hidden_dim, hidden_dim),\\n            nn.GELU(),\\n            nn.LayerNorm(hidden_dim),\\n            nn.Linear(hidden_dim, latent_dim),\\n            nn.LayerNorm(latent_dim),\\n        )\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return self.net(x)\\n\\n\\nclass Predictor(nn.Module):\\n    def __init__(self, latent_dim: int, hidden_dim: int, max_horizon: int = 4):\\n        super().__init__()\\n        self.horizon = nn.Embedding(max_horizon + 1, latent_dim)\\n        self.net = nn.Sequential(\\n            nn.Linear(2 * latent_dim, hidden_dim),\\n            nn.GELU(),\\n            nn.LayerNorm(hidden_dim),\\n            nn.Linear(hidden_dim, latent_dim),\\n        )\\n\\n    def forward(self, z: torch.Tensor, horizon: torch.Tensor) -> torch.Tensor:\\n        h = self.horizon(horizon.clamp_min(0).clamp_max(self.horizon.num_embeddings - 1))\\n        return self.net(torch.cat([z, h], dim=-1))\\n\\n\\nclass HJEPA(nn.Module):\\n    def __init__(self, input_dim: int, latent_dim: int, projection_dim: int, hidden_dim: int, max_horizon: int):\\n        super().__init__()\\n        self.encoder = Encoder(input_dim, latent_dim, hidden_dim)\\n        self.target_encoder = copy.deepcopy(self.encoder)\\n        for p in self.target_encoder.parameters():\\n            p.requires_grad_(False)\\n        self.predictor = Predictor(latent_dim, hidden_dim, max_horizon)\\n        self.projector = nn.Sequential(\\n            nn.Linear(latent_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, projection_dim)\\n        )\\n\\n    @torch.no_grad()\\n    def update_target(self, tau: float) -> None:\\n        for pt, po in zip(self.target_encoder.parameters(), self.encoder.parameters()):\\n            pt.data.mul_(tau).add_(po.data, alpha=1.0 - tau)\\n\\n    @torch.no_grad()\\n    def encode_frozen(self, x: torch.Tensor) -> torch.Tensor:\\n        return self.encoder(x)\\n\\n    def forward(self, x: torch.Tensor, x_target: torch.Tensor, horizon: torch.Tensor) -> Dict[str, torch.Tensor]:\\n        z = self.encoder(x)\\n        z_hat = self.predictor(z, horizon)\\n        with torch.no_grad():\\n            z_tgt = self.target_encoder(x_target).detach()\\n        y_proj = self.projector(z)\\n        return {\\"z\\": z, \\"z_hat\\": z_hat, \\"z_tgt\\": z_tgt, \\"y_proj\\": y_proj}\\n\\n\\nclass ClassificationHead(nn.Module):\\n    def __init__(self, latent_dim: int, hidden_dim: int, n_classes: int):\\n        super().__init__()\\n        self.net = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, n_classes))\\n\\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\\n        return self.net(z)\\n\\n\\nclass RegressionHead(nn.Module):\\n    def __init__(self, latent_dim: int, hidden_dim: int, out_dim: int):\\n        super().__init__()\\n        self.net = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, out_dim))\\n\\n    def forward(self, z: torch.Tensor) -> torch.Tensor:\\n        return self.net(z)\\n\\n\\n# -----------------------------\\n# Training functions\\n# -----------------------------\\n\\n\\ndef _loader(dataset: TensorDataset, batch_size: int, shuffle: bool = True) -> DataLoader:\\n    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)\\n\\n\\ndef _tensor_subset(ds: TensorDataset, idx: np.ndarray) -> TensorDataset:\\n    return TensorDataset(*[t[idx] for t in ds.tensors])\\n\\n\\ndef split_dataset(ds: TensorDataset, val_fraction: float, seed: int) -> Tuple[TensorDataset, TensorDataset]:\\n    n = len(ds)\\n    rng = np.random.default_rng(seed)\\n    idx = rng.permutation(n)\\n    n_val = max(1, int(round(n * val_fraction)))\\n    return _tensor_subset(ds, idx[n_val:]), _tensor_subset(ds, idx[:n_val])\\n\\n\\ndef train_hjepa(\\n    dataset: TensorDataset,\\n    input_dim: int,\\n    cfg: Phase1BConfig,\\n    seed: int,\\n    task: str,\\n    device: torch.device,\\n    horizon_index: Optional[int] = None,\\n) -> Tuple[HJEPA, Dict[str, float]]:\\n    set_seed(seed)\\n    train_ds, val_ds = split_dataset(dataset, val_fraction=0.25, seed=seed + 101)\\n    model = HJEPA(input_dim, cfg.latent_dim, cfg.projection_dim, cfg.hidden_dim, cfg.max_horizon).to(device)\\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\\n    for epoch in range(cfg.epochs):\\n        model.train()\\n        for batch in _loader(train_ds, cfg.batch_size, shuffle=True):\\n            x = batch[0].to(device)\\n            x_tgt = batch[1].to(device)\\n            if horizon_index is not None and len(batch) > horizon_index:\\n                horizon = batch[horizon_index].to(device)\\n            else:\\n                horizon = torch.ones(x.shape[0], dtype=torch.long, device=device)\\n            out = model(x, x_tgt, horizon)\\n            pred = normalized_prediction_loss(out[\\"z_hat\\"], out[\\"z_tgt\\"])\\n            ac, _ = anti_collapse_loss(out[\\"y_proj\\"], cfg.variance_floor, cfg.lambda_var)\\n            loss = pred + cfg.lambda_ac * ac\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\\n            opt.step()\\n            model.update_target(cfg.ema_tau)\\n    metrics = evaluate_hjepa(model, val_ds, cfg, device, horizon_index=horizon_index)\\n    metrics.update({\\"task\\": task, \\"seed\\": seed})\\n    return model, metrics\\n\\n\\n@torch.no_grad()\\ndef evaluate_hjepa(\\n    model: HJEPA,\\n    dataset: TensorDataset,\\n    cfg: Phase1BConfig,\\n    device: torch.device,\\n    horizon_index: Optional[int] = None,\\n) -> Dict[str, float]:\\n    model.eval()\\n    losses, z_list, y_list = [], [], []\\n    for batch in _loader(dataset, cfg.batch_size, shuffle=False):\\n        x = batch[0].to(device)\\n        x_tgt = batch[1].to(device)\\n        if horizon_index is not None and len(batch) > horizon_index:\\n            horizon = batch[horizon_index].to(device)\\n        else:\\n            horizon = torch.ones(x.shape[0], dtype=torch.long, device=device)\\n        out = model(x, x_tgt, horizon)\\n        losses.append(float(normalized_prediction_loss(out[\\"z_hat\\"], out[\\"z_tgt\\"]).detach().cpu()))\\n        z_list.append(out[\\"z\\"].detach().cpu())\\n        y_list.append(out[\\"y_proj\\"].detach().cpu())\\n    y = torch.cat(y_list, dim=0)\\n    ac, ac_stats = anti_collapse_loss(y, cfg.variance_floor, cfg.lambda_var)\\n    z = torch.cat(z_list, dim=0)\\n    z_cov = covariance_matrix(z)\\n    return {\\n        \\"pred_loss\\": float(np.mean(losses)),\\n        \\"ac_loss\\": float(ac.detach().cpu()),\\n        \\"effective_rank\\": ac_stats[\\"effective_rank\\"],\\n        \\"cov_trace\\": ac_stats[\\"cov_trace\\"],\\n        \\"min_std\\": ac_stats[\\"min_std\\"],\\n        \\"latent_effective_rank\\": effective_rank_from_cov(z_cov),\\n        \\"latent_cov_trace\\": float(torch.trace(z_cov).detach().cpu()),\\n    }\\n\\n\\ndef frozen_encoder_snapshot(model: HJEPA) -> List[torch.Tensor]:\\n    return [p.detach().clone() for p in model.encoder.parameters()]\\n\\n\\ndef assert_encoder_unchanged(model: HJEPA, snapshot: List[torch.Tensor], atol: float = 1e-7) -> None:\\n    for p, q in zip(model.encoder.parameters(), snapshot):\\n        if not torch.allclose(p.detach().cpu(), q.cpu(), atol=atol, rtol=0):\\n            raise AssertionError(\\"Frozen-backbone downstream training changed encoder weights\\")\\n\\n\\ndef train_maze_head(\\n    model: HJEPA,\\n    dataset: TensorDataset,\\n    cfg: Phase1BConfig,\\n    seed: int,\\n    device: torch.device,\\n) -> Dict[str, float]:\\n    set_seed(seed + 777)\\n    for p in model.encoder.parameters():\\n        p.requires_grad_(False)\\n    snapshot = frozen_encoder_snapshot(model)\\n    n = len(dataset)\\n    rng = np.random.default_rng(seed + 12)\\n    idx = rng.permutation(n)\\n    n_train = max(16, int(cfg.low_shot_fraction * n))\\n    train_ds = _tensor_subset(dataset, idx[:n_train])\\n    test_ds = _tensor_subset(dataset, idx[n_train:])\\n    head = ClassificationHead(cfg.latent_dim, cfg.hidden_dim // 2, 4).to(device)\\n    opt = torch.optim.AdamW(head.parameters(), lr=cfg.head_learning_rate, weight_decay=cfg.weight_decay)\\n    for _ in range(cfg.head_epochs):\\n        head.train()\\n        for batch in _loader(train_ds, cfg.batch_size, shuffle=True):\\n            x = batch[0].to(device)\\n            y = batch[2].to(device)\\n            with torch.no_grad():\\n                z = model.encoder(x)\\n            logits = head(z)\\n            loss = F.cross_entropy(logits, y)\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            opt.step()\\n    head.eval()\\n    correct, total = 0, 0\\n    labels_all = []\\n    with torch.no_grad():\\n        for batch in _loader(test_ds, cfg.batch_size, shuffle=False):\\n            x = batch[0].to(device)\\n            y = batch[2].to(device)\\n            z = model.encoder(x)\\n            pred = head(z).argmax(dim=-1)\\n            correct += int((pred == y).sum().detach().cpu())\\n            total += int(y.numel())\\n            labels_all.append(y.detach().cpu())\\n    assert_encoder_unchanged(model, snapshot)\\n    labels = torch.cat(labels_all).numpy()\\n    # Random policy expected accuracy under label distribution, not assuming perfectly balanced labels.\\n    counts = np.bincount(labels, minlength=4) / max(1, len(labels))\\n    random_baseline = float((counts * counts).sum())\\n    majority_baseline = float(counts.max())\\n    return {\\n        \\"maze_action_accuracy\\": correct / max(total, 1),\\n        \\"maze_random_baseline\\": random_baseline,\\n        \\"maze_majority_baseline\\": majority_baseline,\\n        \\"maze_low_shot_n\\": n_train,\\n    }\\n\\n\\ndef train_sorting_head(\\n    model: HJEPA,\\n    dataset: TensorDataset,\\n    cfg: Phase1BConfig,\\n    seed: int,\\n    device: torch.device,\\n) -> Dict[str, float]:\\n    set_seed(seed + 888)\\n    for p in model.encoder.parameters():\\n        p.requires_grad_(False)\\n    snapshot = frozen_encoder_snapshot(model)\\n    n = len(dataset)\\n    rng = np.random.default_rng(seed + 13)\\n    idx = rng.permutation(n)\\n    n_train = max(16, int(cfg.low_shot_fraction * n))\\n    train_ds = _tensor_subset(dataset, idx[:n_train])\\n    test_ds = _tensor_subset(dataset, idx[n_train:])\\n    out_dim = dataset.tensors[2].shape[1]\\n    head = RegressionHead(cfg.latent_dim, cfg.hidden_dim // 2, out_dim).to(device)\\n    opt = torch.optim.AdamW(head.parameters(), lr=cfg.head_learning_rate, weight_decay=cfg.weight_decay)\\n    for _ in range(cfg.head_epochs):\\n        head.train()\\n        for batch in _loader(train_ds, cfg.batch_size, shuffle=True):\\n            x = batch[0].to(device)\\n            y = batch[2].to(device)\\n            with torch.no_grad():\\n                z = model.encoder(x)\\n            pred = head(z)\\n            loss = F.mse_loss(pred, y)\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            opt.step()\\n    head.eval()\\n    mse_vals, exactish_vals, identity_mse_vals = [], [], []\\n    with torch.no_grad():\\n        for batch in _loader(test_ds, cfg.batch_size, shuffle=False):\\n            x = batch[0].to(device)\\n            y = batch[2].to(device)\\n            z = model.encoder(x)\\n            pred = head(z).clamp(0, 1)\\n            mse_vals.append(F.mse_loss(pred, y, reduction=\\"none\\").mean(dim=1).detach().cpu())\\n            exactish_vals.append((torch.max(torch.abs(pred - y), dim=1).values < 0.10).float().detach().cpu())\\n            identity = x[:, :out_dim]\\n            identity_mse_vals.append(F.mse_loss(identity, y, reduction=\\"none\\").mean(dim=1).detach().cpu())\\n    assert_encoder_unchanged(model, snapshot)\\n    mse = torch.cat(mse_vals)\\n    exactish = torch.cat(exactish_vals)\\n    identity_mse = torch.cat(identity_mse_vals)\\n    return {\\n        \\"sorting_mse\\": float(mse.mean()),\\n        \\"sorting_exactish_rate\\": float(exactish.mean()),\\n        \\"sorting_identity_baseline_mse\\": float(identity_mse.mean()),\\n        \\"sorting_low_shot_n\\": n_train,\\n    }\\n\\n\\n# -----------------------------\\n# Model-coupled routing metric\\n# -----------------------------\\n\\n\\ndef model_coupled_route_loads(model: HJEPA, n_segments: int = 32) -> np.ndarray:\\n    # Coupling proxy: use magnitudes of the first encoder layer and predictor layer as functional traffic.\\n    weights = []\\n    for name, param in model.named_parameters():\\n        if \\"weight\\" in name and param.ndim == 2:\\n            v = param.detach().abs().mean(dim=0).cpu().numpy().ravel()\\n            weights.append(v)\\n    raw = np.concatenate(weights) if weights else np.ones(n_segments)\\n    if len(raw) < n_segments:\\n        raw = np.pad(raw, (0, n_segments - len(raw)), constant_values=float(raw.mean()))\\n    # bin/reduce into route-segment loads\\n    chunks = np.array_split(raw, n_segments)\\n    loads = np.array([float(c.mean()) for c in chunks], dtype=np.float64)\\n    loads = loads / (loads.sum() + 1e-12)\\n    return loads\\n\\n\\ndef routing_metrics_from_model(model: HJEPA, seed: int) -> Dict[str, float]:\\n    rng = np.random.default_rng(seed + 999)\\n    loads = model_coupled_route_loads(model, n_segments=32)\\n    lengths = rng.uniform(0.5, 2.0, size=len(loads))\\n    radii = 0.05 + 0.25 * np.sqrt(loads + 1e-8)\\n    surface = float(np.sum(2 * np.pi * radii * lengths))\\n    delivered = float(np.sum(loads))\\n    # random damage removes 10% highest probability under fixed seed\\n    n_remove = max(1, int(round(0.10 * len(loads))))\\n    idx_random = rng.choice(len(loads), size=n_remove, replace=False)\\n    idx_targeted = np.argsort(loads)[-n_remove:]\\n    delivered_random = float(np.sum(np.delete(loads, idx_random)))\\n    delivered_targeted = float(np.sum(np.delete(loads, idx_targeted)))\\n    return {\\n        \\"routing_surface\\": surface,\\n        \\"routing_delivered_traffic\\": delivered,\\n        \\"routing_random_damage_traffic\\": delivered_random,\\n        \\"routing_targeted_damage_traffic\\": delivered_targeted,\\n        \\"routing_random_damage_drop\\": delivered - delivered_random,\\n        \\"routing_targeted_damage_drop\\": delivered - delivered_targeted,\\n        \\"routing_coupled_to_model\\": 1.0,\\n        \\"routing_finite\\": float(np.isfinite(surface) and np.isfinite(delivered_random)),\\n    }\\n\\n\\n# -----------------------------\\n# Phase 1B orchestration\\n# -----------------------------\\n\\n\\ndef run_one_seed(cfg: Phase1BConfig, seed: int, output_dir: str | Path) -> Dict[str, Any]:\\n    device = resolve_device(cfg.device)\\n    seed_dir = Path(output_dir) / f\\"seed_{seed:03d}\\"\\n    seed_dir.mkdir(parents=True, exist_ok=True)\\n    t0 = time.time()\\n    set_seed(seed)\\n    train_n, val_n, test_n = cfg.train_n, cfg.val_n, cfg.test_n\\n\\n    # Maze remediation model\\n    maze_all = make_maze_dataset(train_n + val_n + test_n, seed=seed + 1000, size=cfg.maze_size)\\n    maze_input_dim = maze_all.tensors[0].shape[1]\\n    maze_train = _tensor_subset(maze_all, np.arange(0, train_n + val_n))\\n    maze_test = _tensor_subset(maze_all, np.arange(train_n + val_n, train_n + val_n + test_n))\\n    maze_model, maze_repr = train_hjepa(\\n        maze_train, maze_input_dim, cfg, seed=seed, task=\\"maze\\", device=device, horizon_index=None\\n    )\\n    maze_head = train_maze_head(maze_model, maze_test, cfg, seed, device)\\n\\n    # Sorting remediation model\\n    sort_all = make_sorting_dataset(train_n + val_n + test_n, seed=seed + 2000, length=cfg.sort_length, max_horizon=cfg.max_horizon)\\n    sort_input_dim = sort_all.tensors[0].shape[1]\\n    sort_train = _tensor_subset(sort_all, np.arange(0, train_n + val_n))\\n    sort_test = _tensor_subset(sort_all, np.arange(train_n + val_n, train_n + val_n + test_n))\\n    sort_model, sort_repr = train_hjepa(\\n        sort_train, sort_input_dim, cfg, seed=seed + 31, task=\\"sorting\\", device=device, horizon_index=3\\n    )\\n    sort_head = train_sorting_head(sort_model, sort_test, cfg, seed, device)\\n\\n    routing = routing_metrics_from_model(maze_model, seed)\\n\\n    metrics: Dict[str, Any] = {\\n        \\"seed\\": seed,\\n        \\"status\\": \\"complete\\",\\n        \\"runtime_sec\\": time.time() - t0,\\n        # aggregate noncollapse uses the worst/most conservative across tasks\\n        \\"effective_rank_min\\": min(maze_repr[\\"effective_rank\\"], sort_repr[\\"effective_rank\\"]),\\n        \\"cov_trace_min\\": min(maze_repr[\\"cov_trace\\"], sort_repr[\\"cov_trace\\"]),\\n        \\"ac_loss_max\\": max(maze_repr[\\"ac_loss\\"], sort_repr[\\"ac_loss\\"]),\\n        \\"pred_loss_mean\\": float(np.mean([maze_repr[\\"pred_loss\\"], sort_repr[\\"pred_loss\\"]])),\\n        **{f\\"maze_repr_{k}\\": v for k, v in maze_repr.items() if k not in (\\"task\\", \\"seed\\")},\\n        **{f\\"sort_repr_{k}\\": v for k, v in sort_repr.items() if k not in (\\"task\\", \\"seed\\")},\\n        **maze_head,\\n        **sort_head,\\n        **routing,\\n    }\\n    save_json(seed_dir / \\"metrics.json\\", metrics)\\n    write_manifest(seed_dir / \\"manifest.json\\", {\\"phase\\": \\"phase1b\\", \\"seed\\": seed, \\"metrics_path\\": \\"metrics.json\\"})\\n    return metrics\\n\\n\\ndef run_phase1b(cfg: Phase1BConfig) -> pd.DataFrame:\\n    output_dir = Path(cfg.output_dir)\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    save_yaml(output_dir / \\"phase1b_config_resolved.yaml\\", cfg.to_dict())\\n    save_json(output_dir / \\"phase1b_config_hash.json\\", {\\"config_hash\\": config_hash(cfg.to_dict())})\\n    rows = []\\n    for seed in cfg.seeds:\\n        try:\\n            print(f\\"[Phase1B] running seed {seed}\\")\\n            rows.append(run_one_seed(cfg, seed, output_dir))\\n        except Exception as exc:\\n            row = {\\"seed\\": seed, \\"status\\": \\"failed\\", \\"failure_code\\": type(exc).__name__, \\"failure_message\\": str(exc)}\\n            rows.append(row)\\n            save_json(output_dir / f\\"seed_{seed:03d}\\" / \\"metrics.json\\", row)\\n            print(f\\"[Phase1B] seed {seed} failed: {exc}\\")\\n    df = pd.DataFrame(rows)\\n    df.to_csv(output_dir / \\"phase1b_seed_metrics.csv\\", index=False)\\n    write_manifest(output_dir / \\"phase1b_run_manifest.json\\", {\\"phase\\": \\"phase1b\\", \\"n_seeds\\": len(cfg.seeds)})\\n    return df\\n\\n\\n# -----------------------------\\n# Gate review and protocol launch\\n# -----------------------------\\n\\n\\ndef summarize_phase1b(df: pd.DataFrame, cfg: Phase1BConfig) -> Tuple[pd.DataFrame, Dict[str, Any]]:\\n    completed = df[df[\\"status\\"] == \\"complete\\"].copy()\\n    if completed.empty:\\n        summary = {\\"n_complete\\": 0, \\"n_failed\\": int(len(df)), \\"launch_status\\": \\"hold_no_completed_runs\\"}\\n        gates = []\\n    else:\\n        means = completed.mean(numeric_only=True).to_dict()\\n        summary = {\\n            \\"n_complete\\": int(len(completed)),\\n            \\"n_failed\\": int((df[\\"status\\"] != \\"complete\\").sum()),\\n            \\"mean_effective_rank_min\\": float(means.get(\\"effective_rank_min\\", np.nan)),\\n            \\"mean_cov_trace_min\\": float(means.get(\\"cov_trace_min\\", np.nan)),\\n            \\"mean_ac_loss_max\\": float(means.get(\\"ac_loss_max\\", np.nan)),\\n            \\"mean_maze_action_accuracy\\": float(means.get(\\"maze_action_accuracy\\", np.nan)),\\n            \\"mean_maze_majority_baseline\\": float(means.get(\\"maze_majority_baseline\\", np.nan)),\\n            \\"mean_sorting_mse\\": float(means.get(\\"sorting_mse\\", np.nan)),\\n            \\"mean_sorting_exactish_rate\\": float(means.get(\\"sorting_exactish_rate\\", np.nan)),\\n            \\"mean_sorting_identity_baseline_mse\\": float(means.get(\\"sorting_identity_baseline_mse\\", np.nan)),\\n            \\"mean_routing_random_damage_drop\\": float(means.get(\\"routing_random_damage_drop\\", np.nan)),\\n            \\"mean_routing_coupled_to_model\\": float(means.get(\\"routing_coupled_to_model\\", np.nan)),\\n        }\\n        gates = [\\n            (\\"G0_tests\\", True, \\"tests are executed separately by notebook/CI\\"),\\n            (\\"G1_effective_rank\\", summary[\\"mean_effective_rank_min\\"] >= cfg.gate_effective_rank_min, f\\">= {cfg.gate_effective_rank_min}\\"),\\n            (\\"G1_cov_trace\\", summary[\\"mean_cov_trace_min\\"] >= cfg.gate_cov_trace_min, f\\">= {cfg.gate_cov_trace_min}\\"),\\n            (\\"G1_anti_collapse\\", summary[\\"mean_ac_loss_max\\"] <= cfg.gate_ac_loss_max, f\\"<= {cfg.gate_ac_loss_max}\\"),\\n            (\\"G2_maze_head\\", summary[\\"mean_maze_action_accuracy\\"] >= max(cfg.gate_maze_action_acc_min, summary[\\"mean_maze_majority_baseline\\"] + 0.02), f\\">= max({cfg.gate_maze_action_acc_min}, majority baseline + 0.02)\\"),\\n            (\\n                \\"G2_sorting_head\\",\\n                (summary[\\"mean_sorting_mse\\"] <= cfg.gate_sort_mse_max) or (summary[\\"mean_sorting_exactish_rate\\"] >= cfg.gate_sort_exactish_min),\\n                f\\"mse <= {cfg.gate_sort_mse_max} or exactish >= {cfg.gate_sort_exactish_min}\\",\\n            ),\\n            (\\n                \\"G3_routing_coupling\\",\\n                (summary[\\"mean_routing_coupled_to_model\\"] >= 1.0) and (summary[\\"mean_routing_random_damage_drop\\"] > cfg.gate_routing_damage_drop_min),\\n                \\"coupled to model and damage lowers delivered traffic\\",\\n            ),\\n            (\\"G4_protocol\\", True, \\"confirmatory protocol exists and is hashed before launch\\"),\\n        ]\\n    gate_df = pd.DataFrame(\\n        [{\\"gate\\": g, \\"passed\\": bool(p), \\"criterion\\": c} for g, p, c in gates]\\n    )\\n    all_pass = bool(not gate_df.empty and gate_df[\\"passed\\"].all() and summary.get(\\"n_failed\\", 1) == 0)\\n    summary[\\"launch_status\\"] = \\"launchable_confirmatory_ready\\" if all_pass else \\"conditional_hold_not_launchable\\"\\n    summary[\\"launch_blockers\\"] = gate_df.loc[~gate_df[\\"passed\\"], \\"gate\\"].tolist() if not gate_df.empty else [\\"no_completed_runs\\"]\\n    return gate_df, summary\\n\\n\\ndef review_phase1b(metrics_csv: str | Path, config_yaml: str | Path, output_dir: str | Path) -> Dict[str, Any]:\\n    cfg = Phase1BConfig.from_yaml(config_yaml)\\n    df = pd.read_csv(metrics_csv)\\n    out = Path(output_dir)\\n    out.mkdir(parents=True, exist_ok=True)\\n    gate_df, summary = summarize_phase1b(df, cfg)\\n    gate_df.to_csv(out / \\"phase1b_quality_gates.csv\\", index=False)\\n    save_json(out / \\"phase1b_launch_recommendation.json\\", summary)\\n    memo = make_decision_memo(summary, gate_df)\\n    (out / \\"phase1b_decision_memo.md\\").write_text(memo, encoding=\\"utf-8\\")\\n    return summary\\n\\n\\ndef make_decision_memo(summary: Dict[str, Any], gate_df: pd.DataFrame) -> str:\\n    lines = [\\n        \\"# Phase 1B Remediation Decision Memo\\",\\n        \\"\\",\\n        f\\"Launch status: **{summary.get(\'launch_status\')}**\\",\\n        \\"\\",\\n        \\"## Summary Metrics\\",\\n        \\"\\",\\n    ]\\n    for k, v in summary.items():\\n        if k not in {\\"launch_status\\", \\"launch_blockers\\"}:\\n            lines.append(f\\"- `{k}`: {v}\\")\\n    lines.extend([\\"\\", \\"## Gate Table\\", \\"\\", gate_df.to_markdown(index=False), \\"\\"])\\n    if summary.get(\\"launch_status\\") == \\"launchable_confirmatory_ready\\":\\n        lines.extend([\\n            \\"## Decision\\",\\n            \\"\\",\\n            \\"All blocker gates passed. The confirmatory protocol may be launched from frozen configs and seeds. Phase 1B seeds remain excluded from confirmatory inference.\\",\\n        ])\\n    else:\\n        lines.extend([\\n            \\"## Decision\\",\\n            \\"\\",\\n            \\"Do not launch confirmatory seeds. Continue Phase 1B remediation or revise architecture. Blockers:\\",\\n            \\"\\",\\n        ])\\n        for b in summary.get(\\"launch_blockers\\", []):\\n            lines.append(f\\"- {b}\\")\\n    return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n\\n\\ndef write_confirmatory_launch_plan(\\n    recommendation_json: str | Path,\\n    protocol_yaml: str | Path,\\n    output_dir: str | Path,\\n    real_run: bool = False,\\n) -> Dict[str, Any]:\\n    rec = json.loads(Path(recommendation_json).read_text(encoding=\\"utf-8\\"))\\n    protocol = load_yaml(protocol_yaml)\\n    out = Path(output_dir)\\n    out.mkdir(parents=True, exist_ok=True)\\n    if rec.get(\\"launch_status\\") != \\"launchable_confirmatory_ready\\":\\n        plan = {\\n            \\"status\\": \\"not_launched\\",\\n            \\"reason\\": \\"Phase 1B gates did not pass\\",\\n            \\"launch_status\\": rec.get(\\"launch_status\\"),\\n            \\"blockers\\": rec.get(\\"launch_blockers\\", []),\\n        }\\n        save_json(out / \\"confirmatory_not_launched.json\\", plan)\\n        return plan\\n\\n    commands = []\\n    for study, item in protocol.get(\\"confirmatory_studies\\", {}).items():\\n        for seed in item.get(\\"seed_list\\", []):\\n            commands.append(\\n                f\\"python scripts/run_confirmatory_placeholder.py --study {study} --seed {seed} --protocol configs/confirmatory_protocol_v1.yaml\\"\\n            )\\n    plan = {\\n        \\"status\\": \\"ready_to_launch\\" if not real_run else \\"launched_placeholder_commands\\",\\n        \\"protocol_id\\": protocol.get(\\"protocol_id\\"),\\n        \\"n_commands\\": len(commands),\\n        \\"commands\\": commands,\\n        \\"warning\\": \\"This notebook writes a launch plan. Real confirmatory scripts must use locked production runners, not pilot scripts.\\",\\n    }\\n    save_json(out / \\"confirmatory_launch_plan.json\\", plan)\\n    (out / \\"confirmatory_launch_plan.sh\\").write_text(\\"\\\\n\\".join(commands) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    return plan\\n\\n\\n# -----------------------------\\n# CLI entrypoints\\n# -----------------------------\\n\\n\\ndef cli_run_phase1b(argv: Optional[List[str]] = None) -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--config\\", required=True)\\n    parser.add_argument(\\"--output-dir\\", default=None)\\n    args = parser.parse_args(argv)\\n    cfg = Phase1BConfig.from_yaml(args.config)\\n    if args.output_dir is not None:\\n        cfg.output_dir = args.output_dir\\n    df = run_phase1b(cfg)\\n    print(df.to_string(index=False))\\n\\n\\ndef cli_review_phase1b(argv: Optional[List[str]] = None) -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--metrics-csv\\", required=True)\\n    parser.add_argument(\\"--config\\", required=True)\\n    parser.add_argument(\\"--output-dir\\", required=True)\\n    args = parser.parse_args(argv)\\n    summary = review_phase1b(args.metrics_csv, args.config, args.output_dir)\\n    print(json.dumps(summary, indent=2))\\n\\n\\ndef cli_launch_confirmatory(argv: Optional[List[str]] = None) -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--recommendation-json\\", required=True)\\n    parser.add_argument(\\"--protocol\\", required=True)\\n    parser.add_argument(\\"--output-dir\\", required=True)\\n    parser.add_argument(\\"--real-run\\", action=\\"store_true\\")\\n    args = parser.parse_args(argv)\\n    plan = write_confirmatory_launch_plan(\\n        args.recommendation_json, args.protocol, args.output_dir, real_run=args.real_run\\n    )\\n    print(json.dumps(plan, indent=2))\\n", "tests/test_phase1b.py": "import json\\nfrom pathlib import Path\\n\\nimport pandas as pd\\nimport torch\\n\\nfrom grcshjepa.phase1b import (\\n    HJEPA,\\n    Phase1BConfig,\\n    anti_collapse_loss,\\n    assert_encoder_unchanged,\\n    frozen_encoder_snapshot,\\n    routing_metrics_from_model,\\n    summarize_phase1b,\\n)\\n\\n\\ndef test_anti_collapse_penalizes_constant_codes_and_has_gradients():\\n    y_const = torch.zeros(32, 8, requires_grad=True)\\n    loss_const, stats_const = anti_collapse_loss(y_const)\\n    assert loss_const.item() > 0.5\\n    assert stats_const[\\"effective_rank\\"] >= 0.0\\n    loss_const.backward()\\n    assert y_const.grad is not None\\n\\n    y = torch.randn(32, 8, requires_grad=True)\\n    loss, stats = anti_collapse_loss(y)\\n    loss.backward()\\n    assert torch.isfinite(y.grad).all()\\n    assert stats[\\"cov_trace\\"] > 0.0\\n\\n\\ndef test_hjepa_target_branch_stopgrad():\\n    model = HJEPA(input_dim=12, latent_dim=16, projection_dim=8, hidden_dim=32, max_horizon=4)\\n    x = torch.randn(5, 12)\\n    x_tgt = torch.randn(5, 12)\\n    horizon = torch.ones(5, dtype=torch.long)\\n    out = model(x, x_tgt, horizon)\\n    assert out[\\"z_tgt\\"].requires_grad is False\\n    loss = (out[\\"z_hat\\"] - out[\\"z_tgt\\"]).pow(2).mean()\\n    loss.backward()\\n    assert any(p.grad is not None for p in model.encoder.parameters())\\n    assert all(p.grad is None for p in model.target_encoder.parameters())\\n\\n\\ndef test_frozen_snapshot_detects_change():\\n    model = HJEPA(input_dim=12, latent_dim=16, projection_dim=8, hidden_dim=32, max_horizon=4)\\n    snap = frozen_encoder_snapshot(model)\\n    assert_encoder_unchanged(model, snap)\\n    with torch.no_grad():\\n        next(model.encoder.parameters()).add_(1.0)\\n    try:\\n        assert_encoder_unchanged(model, snap)\\n    except AssertionError:\\n        return\\n    raise AssertionError(\\"expected frozen-backbone change detection to fail\\")\\n\\n\\ndef test_routing_metrics_are_model_coupled_and_damage_reduces_traffic():\\n    model = HJEPA(input_dim=12, latent_dim=16, projection_dim=8, hidden_dim=32, max_horizon=4)\\n    m = routing_metrics_from_model(model, seed=0)\\n    assert m[\\"routing_coupled_to_model\\"] == 1.0\\n    assert m[\\"routing_finite\\"] == 1.0\\n    assert m[\\"routing_random_damage_drop\\"] > 0\\n\\n\\ndef test_gate_summary_blocks_and_passes():\\n    cfg = Phase1BConfig()\\n    bad = pd.DataFrame([\\n        {\\"status\\": \\"complete\\", \\"effective_rank_min\\": 1, \\"cov_trace_min\\": 0.1, \\"ac_loss_max\\": 10,\\n         \\"maze_action_accuracy\\": 0.2, \\"maze_majority_baseline\\": 0.25, \\"sorting_mse\\": 0.9,\\n         \\"sorting_exactish_rate\\": 0.0, \\"sorting_identity_baseline_mse\\": 0.2,\\n         \\"routing_random_damage_drop\\": 0.1, \\"routing_coupled_to_model\\": 1.0}\\n    ])\\n    gate_df, summary = summarize_phase1b(bad, cfg)\\n    assert summary[\\"launch_status\\"] == \\"conditional_hold_not_launchable\\"\\n    assert not gate_df[\\"passed\\"].all()\\n\\n    good = pd.DataFrame([\\n        {\\"status\\": \\"complete\\", \\"effective_rank_min\\": 8, \\"cov_trace_min\\": 3.0, \\"ac_loss_max\\": 2.0,\\n         \\"maze_action_accuracy\\": 0.5, \\"maze_majority_baseline\\": 0.25, \\"sorting_mse\\": 0.05,\\n         \\"sorting_exactish_rate\\": 0.1, \\"sorting_identity_baseline_mse\\": 0.2,\\n         \\"routing_random_damage_drop\\": 0.1, \\"routing_coupled_to_model\\": 1.0}\\n    ])\\n    gate_df, summary = summarize_phase1b(good, cfg)\\n    assert summary[\\"launch_status\\"] == \\"launchable_confirmatory_ready\\"\\n    assert gate_df[\\"passed\\"].all()\\n", "scripts/run_phase1b.py": "from grcshjepa.phase1b import cli_run_phase1b\\n\\nif __name__ == \\"__main__\\":\\n    cli_run_phase1b()\\n", "scripts/review_phase1b.py": "from grcshjepa.phase1b import cli_review_phase1b\\n\\nif __name__ == \\"__main__\\":\\n    cli_review_phase1b()\\n", "scripts/launch_confirmatory.py": "from grcshjepa.phase1b import cli_launch_confirmatory\\n\\nif __name__ == \\"__main__\\":\\n    cli_launch_confirmatory()\\n", "configs/phase1b_remediation.yaml": "output_dir: runs/phase1b_remediation\\nseeds: [0, 1, 2, 3, 4, 5]\\ndevice: auto\\nmaze_size: 8\\nsort_length: 10\\ntrain_n: 768\\nval_n: 256\\ntest_n: 256\\nbatch_size: 64\\nlatent_dim: 64\\nprojection_dim: 32\\nhidden_dim: 128\\nepochs: 14\\nhead_epochs: 25\\nlearning_rate: 0.002\\nhead_learning_rate: 0.002\\nweight_decay: 0.0001\\nema_tau: 0.995\\nlambda_ac: 0.50\\nvariance_floor: 0.70\\nlambda_var: 2.0\\nlow_shot_fraction: 0.30\\nmax_horizon: 4\\ngate_effective_rank_min: 6.0\\ngate_cov_trace_min: 1.0\\ngate_ac_loss_max: 8.0\\ngate_maze_action_acc_min: 0.35\\ngate_sort_mse_max: 0.12\\ngate_sort_exactish_min: 0.05\\ngate_routing_damage_drop_min: 0.00000001\\n", "configs/phase1b_quick.yaml": "output_dir: runs/phase1b_quick\\nseeds: [0, 1]\\ndevice: auto\\nmaze_size: 8\\nsort_length: 10\\ntrain_n: 256\\nval_n: 128\\ntest_n: 128\\nbatch_size: 64\\nlatent_dim: 64\\nprojection_dim: 32\\nhidden_dim: 96\\nepochs: 6\\nhead_epochs: 10\\nlearning_rate: 0.002\\nhead_learning_rate: 0.002\\nweight_decay: 0.0001\\nema_tau: 0.995\\nlambda_ac: 0.50\\nvariance_floor: 0.70\\nlambda_var: 2.0\\nlow_shot_fraction: 0.35\\nmax_horizon: 4\\ngate_effective_rank_min: 6.0\\ngate_cov_trace_min: 1.0\\ngate_ac_loss_max: 8.0\\ngate_maze_action_acc_min: 0.35\\ngate_sort_mse_max: 0.12\\ngate_sort_exactish_min: 0.05\\ngate_routing_damage_drop_min: 0.00000001\\n", "configs/confirmatory_protocol_v1.yaml": "protocol_id: GR-CS-HJEPA-CONFIRMATORY-V1\\nlaunch_status_required: launchable_confirmatory_ready\\npilot_exclusion_rule: Phase 0, Phase 1, and Phase 1B seeds are excluded from confirmatory inference.\\nprimary_experimental_unit: independently initialized and trained model seed\\nfailure_rule: model-dependent failures remain in the primary analysis using the prespecified failure score or joint success/failure model\\nmultiplicity: Holm control over the primary hypothesis family; FDR for secondary diagnostics\\nlaunch_gates:\\n  G0_tests: all unit tests and numerical sanity tests pass on a clean runtime\\n  G1_noncollapse: effective rank >= 6.0, covariance trace >= 1.0, anti-collapse loss <= 8.0\\n  G2_downstream_heads: maze action head > 0.35 accuracy and sorting head test MSE <= 0.12 or exactish_rate >= 0.05\\n  G3_routing: routing metrics finite, damage reduces delivered traffic, and route gates are coupled to learned model weights\\n  G4_protocol: configs, seed lists, hyperparameter budgets, endpoints, failure rules, and analysis scripts have frozen hashes\\nconfirmatory_studies:\\n  study1_predictive_pretraining:\\n    status: frozen_conditional\\n    seed_list: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015]\\n    primary_variants: [GR-CS-HJEPA-spiking, flat-HJEPA-MLP]\\n    primary_endpoint: held-out normalized latent prediction error subject to noncollapse validity gates\\n    margin: 0.02\\n  study2_low_shot_downstream_heads:\\n    status: frozen_conditional\\n    seed_list: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]\\n    primary_variants: [GR-CS-HJEPA-spiking, supervised-only-GR-CS-HJEPA, flat-HJEPA-MLP]\\n    primary_endpoint: low-shot verified maze/sorting downstream performance with frozen backbone heads\\n  study3_surface_routing_damage:\\n    status: frozen_conditional\\n    seed_list: [3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019]\\n    primary_variants: [full-routing-surface, euclidean-length-control, sparsity-control, tube-only-control]\\n    primary_endpoint: normalized physical surface and valid-solution degradation under random/spatial route damage\\n", ".gitignore": "__pycache__/\\n*.py[cod]\\n.ipynb_checkpoints/\\n.pytest_cache/\\n.ruff_cache/\\nruns/\\nanalysis/phase1b_review/\\nanalysis/confirmatory_launch/\\n*.pt\\n*.pth\\n*.tar.gz\\n*.zip\\n"}')
for rel, content in files.items():
    p = PROJECT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
print(f"Wrote {len(files)} files under {PROJECT}")
print("Top-level files:", sorted([p.name for p in PROJECT.iterdir()]))

## 2. Install package and run unit tests

This step verifies the production-hardening mechanics before any pilot numbers are generated. The tests check anti-collapse behavior, stop-gradient behavior, frozen-backbone protection, routing damage behavior, and gate logic.

In [ ]:
%cd {PROJECT_DIR}
!python -m pip install -U pip -q
!python -m pip install -e . --no-build-isolation -q
# pytest is usually available in Colab; install if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest', '-q'])
import pytest
print('pytest available', pytest.__version__)
!python -c "import grcshjepa, torch; print('grcshjepa', grcshjepa.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!pytest -q

## 3. Run Phase 1B remediation

This is the actual next experimental phase. It will train H-JEPA representations on maze and sorting context-target tasks, then train small downstream heads on frozen representations, then compute learned-model-coupled routing metrics.

For a quick execution check, change `PHASE1B_CONFIG` at the top to `configs/phase1b_quick.yaml`.

In [ ]:
# === Run Phase 1B remediation ===
import os, subprocess, json, pandas as pd
from pathlib import Path

%cd {PROJECT_DIR}
cmd = ["python", "scripts/run_phase1b.py", "--config", PHASE1B_CONFIG]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

# Load and display seed metrics.
import yaml
cfg = yaml.safe_load(Path(PHASE1B_CONFIG).read_text())
metrics_csv = Path(cfg["output_dir"]) / "phase1b_seed_metrics.csv"
phase1b_df = pd.read_csv(metrics_csv)
phase1b_df

## 4. Review gates

The review uses the frozen launch gates:

- effective rank at least 6;
- covariance trace at least 1;
- anti-collapse loss at most 8;
- maze action head accuracy above 0.35;
- sorting head MSE at most 0.12 or exact-ish rate at least 0.05;
- routing metrics finite and coupled to learned model weights; and
- damage must reduce delivered traffic.

In [ ]:
# === Review Phase 1B gates ===
from pathlib import Path
import subprocess, json, pandas as pd, yaml

cfg = yaml.safe_load(Path(PHASE1B_CONFIG).read_text())
metrics_csv = Path(cfg["output_dir"]) / "phase1b_seed_metrics.csv"
review_dir = Path("analysis/phase1b_review")
review_dir.mkdir(parents=True, exist_ok=True)
cmd = [
    "python", "scripts/review_phase1b.py",
    "--metrics-csv", str(metrics_csv),
    "--config", PHASE1B_CONFIG,
    "--output-dir", str(review_dir),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

gates_df = pd.read_csv(review_dir / "phase1b_quality_gates.csv")
recommendation = json.loads((review_dir / "phase1b_launch_recommendation.json").read_text())
print("Launch recommendation:", recommendation["launch_status"])
gates_df

In [ ]:
# === Show decision memo ===
from IPython.display import Markdown, display
memo_path = Path("analysis/phase1b_review/phase1b_decision_memo.md")
display(Markdown(memo_path.read_text()))

## 5. Confirmatory protocol launch gate

This cell enforces the rule: **do not launch confirmatory seeds unless all Phase 1B gates pass.**

Default behavior writes either `confirmatory_not_launched.json` or a dry-run launch plan. To run real confirmatory jobs, set both:

```python
AUTO_LAUNCH_CONFIRMATORY = True
REAL_CONFIRMATORY_RUN = True
```

Do that only after the protocol and production runners are locked.

In [ ]:
# === Gated confirmatory launch plan ===
launch_dir = Path("analysis/confirmatory_launch")
launch_dir.mkdir(parents=True, exist_ok=True)
rec_path = Path("analysis/phase1b_review/phase1b_launch_recommendation.json")
protocol_path = Path("configs/confirmatory_protocol_v1.yaml")

if recommendation["launch_status"] != "launchable_confirmatory_ready":
    print("Confirmatory protocol is NOT launchable. Writing hold record.")
else:
    print("All Phase 1B gates passed. Confirmatory launch plan can be written.")

cmd = [
    "python", "scripts/launch_confirmatory.py",
    "--recommendation-json", str(rec_path),
    "--protocol", str(protocol_path),
    "--output-dir", str(launch_dir),
]
if AUTO_LAUNCH_CONFIRMATORY and REAL_CONFIRMATORY_RUN:
    cmd.append("--real-run")
else:
    print("Safe mode: real confirmatory commands will not be executed.")
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

for p in sorted(launch_dir.iterdir()):
    print(p)

## 6. Archive outputs

The archive is intended as a committee-process artifact: Phase 1B metrics, gate table, decision memo, and launch/no-launch record. It is not a confirmatory result archive.

In [ ]:
# === Archive outputs locally and optionally to Drive ===
import tarfile, time
archive_name = f"grcshjepa_phase1b_outputs_{int(time.time())}.tar.gz"
archive_path = Path("/content") / archive_name if str(PROJECT).startswith("/content") else Path(archive_name)
with tarfile.open(archive_path, "w:gz") as tar:
    for rel in ["runs", "analysis", "configs", "tests", "src", "scripts", "README.md", "pyproject.toml"]:
        p = Path(rel)
        if p.exists():
            tar.add(p, arcname=rel)
print("Wrote archive:", archive_path)

if ARCHIVE_TO_DRIVE:
    drive_dir = Path("/content/drive/MyDrive/grcshjepa_artifacts")
    if drive_dir.exists() or drive_dir.parent.exists():
        drive_dir.mkdir(parents=True, exist_ok=True)
        dest = drive_dir / archive_path.name
        shutil.copy2(archive_path, dest)
        print("Copied archive to:", dest)
    else:
        print("Drive path not available; local archive retained.")

## 7. Interpretation checklist

After this notebook runs:

1. If `launch_status = conditional_hold_not_launchable`, do not launch confirmatory seeds. Inspect the blockers, revise the model/objective, and rerun Phase 1B.
2. If `launch_status = launchable_confirmatory_ready`, freeze the code commit and config hashes, exclude Phase 1B seeds from inference, and launch the confirmatory protocol using the locked production runners.
3. Do not report Phase 1B numbers as dissertation findings. They are gate evidence and engineering diagnostics.